In [4]:
import os
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. CONFIGURACIÓN DE LA PETICIÓN REAL (Indicaciones de la Rúbrica)
URL = "https://www.meneame.net/"
# User-Agent ético identificando que es un proyecto académico universitario
headers = {
    "User-Agent": "ProyectoUniversitarioBot/1.0 (Contacto: taller2_desinformacion@universidad.edu.pe)"
}

print(f"Conectando a {URL} de manera segura y respetuosa...")

try:
    response = requests.get(URL, headers=headers, timeout=10)
    response.raise_for_status() # Verifica que la página cargue correctamente (Código 200)
    contenido_html = response.text
except Exception as e:
    print(f"Error al conectar con la web real: {e}")
    print("Asegúrate de tener conexión a internet.")
    exit()

# 2. PARSEO CON BEAUTIFULSOUP
soup = BeautifulSoup(contenido_html, "html.parser")

# En Menéame, cada noticia e historia pública está contenida en un div con la clase "news-summary"
noticias_html = soup.select("div.news-summary")

lista_extraccion = []

print(f"Extrayendo datos de {len(noticias_html)} noticias en vivo...\n")

# 3. EXTRACCIÓN DE LAS VARIABLES EN ENTORNO REAL
for noticia in noticias_html:
    try:
        # Extraer título y enlace (dentro del contenedor principal de texto)
        bloque_titulo = noticia.select_one("div.center-content h2 a")
        if not bloque_titulo:
            continue
            
        titulo = bloque_titulo.get_text().strip()
        enlace = bloque_titulo["href"]
        
        # Extraer el texto descriptivo o entradilla de la noticia
        descripcion_raw = noticia.select_one("div.news-content")
        descripcion = descripcion_raw.get_text().strip() if descripcion_raw else "Sin descripción"
        
        # Extraer métricas de interés (Votos y Clics) para analizar comportamiento anómalo
        votos_raw = noticia.select_one("div.votes a")
        votos_texto = votos_raw.get_text().strip() if votos_raw else "0"
        
        clics_raw = noticia.select_one("div.clics")
        clics_texto = clics_raw.get_text().strip() if clics_raw else "0 clics"
        
        lista_extraccion.append({
            "Título": titulo,
            "Enlace_Origen": enlace,
            "Descripción": descripcion,
            "Votos_Original": votos_texto,
            "Clics_Original": clics_texto
        })
    except Exception as e:
        # Evita que el programa se caiga si una noticia viene con estructura corrupta
        continue

    # Respetar restricciones del servidor: pausa de 0.2 segundos por bucle interno si procesáramos subpáginas
    # (Como aquí jalamos todo el home de un solo golpe, esta pausa protege futuras llamadas)
    time.sleep(0.1)

# 4. CREAR EL DATAFRAME DE PANDAS
df_noticias = pd.DataFrame(lista_extraccion)

# 5. ANÁLISIS DE SENSACIONALISMO / CLICKBAIT (Métrica de Evaluación de la solución)
# Limpiamos las cadenas de texto para convertirlas en números enteros limpios
df_noticias["Votos_Numerico"] = df_noticias["Votos_Original"].str.replace(r"\D", "", regex=True).astype(int)
df_noticias["Clics_Numerico"] = df_noticias["Clics_Original"].str.replace(r"\D", "", regex=True).astype(int)

# Criterio de Sospecha: Si una noticia acumula muchos clics pero la comunidad no la vota/valida
# Calculamos un ratio de Validación. Si los votos representan menos del 3% de los clics entrantes,
# el titular podría estar diseñado exclusivamente como un anzuelo desinformativo o clickbait morboso.
df_noticias["Ratio_Validacion_%"] = (df_noticias["Votos_Numerico"] / (df_noticias["Clics_Numerico"] + 1)) * 100

# Umbral crítico definido para el prototipo
UMBRAL_SOSPECHA = 3.0
df_noticias["¿Es_Sospechosa?"] = df_noticias["Ratio_Validacion_%"] < UMBRAL_SOSPECHA

# Separar registros legítimos de los sospechosos
noticias_legitimas = df_noticias[df_noticias["¿Es_Sospechosa?"] == False]
noticias_sospechosas = df_noticias[df_noticias["¿Es_Sospechosa?"] == True]

# 6. REPORTE ESTADÍSTICO EXIGIDO POR LA RÚBRICA
print("=======================================================")
print("    REPORTE EN TIEMPO REAL: DETECCIÓN DE CLICKBAIT     ")
print("=======================================================")
print(f"Total de noticias raspadas y analizadas : {len(df_noticias)}")
print(f"Noticias validadas u orgánicas         : {len(noticias_legitimas)}")
print(f"Noticias SOSPECHOSAS (Bajo engagement) : {len(noticias_sospechosas)}")
print("=======================================================\n")

print(" ALERTAS: ENLACES ALTAMENTE SENSACIONALISTAS O ANÓMALOS:")
for index, fila in noticias_sospechosas.iterrows():
    print(f"- Alerta: '{fila['Título'][:60]}...' ")
    print(f"  Métricas -> Clics: {fila['Clics_Numerico']} | Votos: {fila['Votos_Numerico']} | Ratio: {fila['Ratio_Validacion_%']:.2f}%")
    print(f"  Enlace  -> {fila['Enlace_Origen']}\n")

# 7. GUARDAR ALMACENAMIENTO DE DATOS CONSOLIDADO (JSON/CSV según rúbrica)
archivo_salida = "reporte_meneame_desinformacion.csv"
df_noticias.to_csv(archivo_salida, index=False, encoding="utf-8")
print(f"✅ Éxito: Datos consolidados exportados a '{archivo_salida}' para repositorio GitHub.")

Conectando a https://www.meneame.net/ de manera segura y respetuosa...
Extrayendo datos de 25 noticias en vivo...

    REPORTE EN TIEMPO REAL: DETECCIÓN DE CLICKBAIT     
Total de noticias raspadas y analizadas : 25
Noticias validadas u orgánicas         : 23
Noticias SOSPECHOSAS (Bajo engagement) : 2

 ALERTAS: ENLACES ALTAMENTE SENSACIONALISTAS O ANÓMALOS:
- Alerta: 'Joan Collins: impactantes retratos del rodaje de «Tierra de ...' 
  Métricas -> Clics: 1594 | Votos: 43 | Ratio: 2.70%
  Enlace  -> http://www.vintag.es/2026/06/joan-collins-land-of-the-pharaohs.html

- Alerta: '"Atropellado en León un anciano de 45 años a la altura de Ma...' 
  Métricas -> Clics: 10477 | Votos: 192 | Ratio: 1.83%
  Enlace  -> https://www.meneame.net/story/atropellado-leon-anciano-45-anos-altura-mariano-andres-tpm

✅ Éxito: Datos consolidados exportados a 'reporte_meneame_desinformacion.csv' para repositorio GitHub.
